<a href="https://colab.research.google.com/github/MayerT1/Prep_GEDI/blob/main/Stage_2_Bake_Off.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os

# Define the base project folder and subdirectories
base_dir = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project'
subdirs = [
    '2024_Imagery_For_Inference'
    'data_32_32_patches_11_4_25',
    'target_data',
    'data',
    'data_NaN_filtered',
    'models',
    "model_animations",
    'scripts',
    'notebooks',
    'config',
    'results',
    'prediction_surface'
]

# Create each subdirectory
for subdir in subdirs:
    path = os.path.join(base_dir, subdir)
    os.makedirs(path, exist_ok=True)
    print(f"Created: {path}")

Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/2024_Imagery_For_Inferencedata_32_32_patches_11_4_25
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/target_data
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/data_NaN_filtered
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/models
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/model_animations
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/scripts
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/notebooks
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/config
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/results
Created: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/prediction_surface


In [ ]:
# # =====================================================
# # STAGE 2 MODEL BAKE-OFF - WITH HYPERPARAMETER TUNING
# # Fair comparison with grid search for all models
# # =====================================================

# import os, sys, time, pickle, warnings
# warnings.filterwarnings('ignore')

# import numpy as np
# import pandas as pd
# from collections import defaultdict
# from itertools import product

# import matplotlib.pyplot as plt
# import seaborn as sns
# from matplotlib.gridspec import GridSpec

# try:
#     import torch
#     import torch.nn as nn
#     from torch.utils.data import Dataset, DataLoader
# except RuntimeError as e:
#     if "TORCH_LIBRARY" in str(e):
#         print("⚠️  RESTART RUNTIME"); sys.exit(1)
#     raise

# from sklearn.linear_model import Ridge, Lasso, ElasticNet
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import ParameterGrid
# from sklearn.metrics import r2_score
# from scipy import stats
# # from scipy.spatial.distance import wasserstein_distance
# from scipy.stats import wasserstein_distance


# print("="*70)
# print("STAGE 2 MODEL BAKE-OFF - WITH HYPERPARAMETER TUNING")
# print("="*70)

# # =====================================================
# # CONFIGURATION
# # =====================================================

# DATA_DIR = "/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/Embedding_Checkpoints_Stage1"
# OUTPUT_DIR = "/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/Stage2_Bakeoff_Results"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# SEED = 42

# # Tuning strategy
# TUNING_METRIC = "coverage"  # Options: "mae", "rmse", "coverage", "composite"
# COMPOSITE_WEIGHTS = {
#     "mae": 0.4,      # 40% accuracy
#     "coverage": 0.4,  # 40% variance preservation
#     "r2": 0.2        # 20% fit quality
# }

# np.random.seed(SEED)
# torch.manual_seed(SEED)
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # =====================================================
# # PRE-FLIGHT CHECKS
# # =====================================================

# print("\n🔍 PRE-FLIGHT CHECKS")
# print("="*70)

# TRAIN_FILE = f"{DATA_DIR}/dataset_train.pt"
# TEST_FILE = f"{DATA_DIR}/dataset_test.pt"
# VAL_FILE = f"{DATA_DIR}/dataset_val.pt"

# checks_passed = True
# for name, path in [("Train", TRAIN_FILE), ("Test", TEST_FILE), ("Val", VAL_FILE)]:
#     if os.path.exists(path):
#         size_mb = os.path.getsize(path) / (1024**2)
#         print(f"   ✓ {name}: {size_mb:.1f} MB")
#     else:
#         print(f"   ✗ {name} NOT FOUND: {path}")
#         checks_passed = False

# print(f"   ✓ Device: {DEVICE}")
# print(f"   ✓ Output: {OUTPUT_DIR}")
# print(f"   ✓ Tuning metric: {TUNING_METRIC}")

# if not checks_passed:
#     print("\n❌ PRE-FLIGHT CHECKS FAILED")
#     sys.exit(1)

# print("\n✅ ALL CHECKS PASSED")
# print("\n⚠️  This will perform hyperparameter tuning + full training.")
# print("    Estimated time: 2-4 hours")
# print("="*70)

# # =====================================================
# # LOAD DATA
# # =====================================================

# print("\nLOADING DATA")
# print("="*70)

# dataset_train = torch.load(TRAIN_FILE, weights_only=False)
# dataset_test = torch.load(TEST_FILE, weights_only=False)
# dataset_val = torch.load(VAL_FILE, weights_only=False)

# def extract_data(dataset):
#     X, y = [], []
#     for s in dataset:
#         X.append(s['embeddings'].flatten().numpy())
#         y.append(s['target'].item())
#     return np.array(X), np.array(y)

# X_train, y_train = extract_data(dataset_train)
# X_test, y_test = extract_data(dataset_test)
# X_val, y_val = extract_data(dataset_val)

# print(f"✓ Train: {X_train.shape} samples")
# print(f"✓ Val:   {X_val.shape} samples")
# print(f"✓ Test:  {X_test.shape} samples")
# print(f"\n✓ Target ranges:")
# print(f"  Train: {y_train.min():.1f}-{y_train.max():.1f}m (std={y_train.std():.1f}m)")
# print(f"  Val:   {y_val.min():.1f}-{y_val.max():.1f}m (std={y_val.std():.1f}m)")
# print(f"  Test:  {y_test.min():.1f}-{y_test.max():.1f}m (std={y_test.std():.1f}m)")

# target_range_val = y_val.max() - y_val.min()
# target_range_test = y_test.max() - y_test.min()

# # =====================================================
# # HYPERPARAMETER GRIDS
# # =====================================================

# print("\n" + "="*70)
# print("DEFINING HYPERPARAMETER SEARCH SPACES")
# print("="*70)

# HYPERPARAMETER_GRIDS = {
#     "RF_Raw_Baseline": {
#         "n_estimators": [50, 100, 200],
#         "max_depth": [5, 10, 15, 20],
#         "min_samples_split": [2, 5, 10],
#         "min_samples_leaf": [1, 2, 4]
#     },

#     "Ridge": {
#         "alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
#     },

#     "Lasso": {
#         "alpha": [0.001, 0.01, 0.1, 1.0, 10.0]
#     },

#     "ElasticNet": {
#         "alpha": [0.1, 1.0, 10.0],
#         "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
#     },

#     "RF_Embeddings": {
#         "n_estimators": [50, 100, 200],
#         "max_depth": [5, 10, 15, 20],
#         "min_samples_split": [2, 5, 10],
#         "min_samples_leaf": [1, 2, 4]
#     },

#     "SimpleMLP": {
#         "hidden_dims": [
#             [2048, 512, 128],
#             [2048, 1024, 512, 128],
#             [4096, 1024, 256]
#         ],
#         "dropout": [0.2, 0.3, 0.4],
#         "lr": [5e-4, 1e-3, 2e-3]
#     },

#     "MoE": {
#         "num_experts": [3, 5, 7],
#         "hidden_dims": [[1024, 512, 256], [2048, 512, 256]],
#         "dropout": [0.2, 0.3],
#         "lr": [5e-4, 1e-3]
#     },

#     "Hierarchical": {
#         "num_layers": [2, 4, 6],
#         "nhead": [4, 8],
#         "dropout": [0.1, 0.2, 0.3],
#         "lr": [5e-4, 1e-3]
#     }
# }

# # Print grid sizes
# print("\nSearch space sizes:")
# for model_name, grid in HYPERPARAMETER_GRIDS.items():
#     param_grid = ParameterGrid(grid)
#     print(f"  {model_name}: {len(param_grid)} combinations")

# total_configs = sum(len(ParameterGrid(grid)) for grid in HYPERPARAMETER_GRIDS.values())
# print(f"\n✓ Total configurations to evaluate: {total_configs}")

# # =====================================================
# # PYTORCH COMPONENTS
# # =====================================================

# class SimpleDataset(Dataset):
#     def __init__(self, X, y):
#         self.X = torch.FloatTensor(X)
#         self.y = torch.FloatTensor(y)
#     def __len__(self): return len(self.X)
#     def __getitem__(self, idx): return self.X[idx], self.y[idx]

# # MODEL ARCHITECTURES (parameterized)

# class SimpleMLP(nn.Module):
#     def __init__(self, hidden_dims=[2048, 512, 128], dropout=0.3):
#         super().__init__()
#         layers = []
#         prev_dim = 65536
#         for hidden_dim in hidden_dims:
#             layers.extend([
#                 nn.Linear(prev_dim, hidden_dim),
#                 nn.ReLU(),
#                 nn.Dropout(dropout)
#             ])
#             prev_dim = hidden_dim
#         layers.append(nn.Linear(prev_dim, 1))
#         self.net = nn.Sequential(*layers)

#     def forward(self, x):
#         return self.net(x).squeeze(-1)

# class MixtureOfExperts(nn.Module):
#     def __init__(self, num_experts=5, hidden_dims=[1024, 512, 256], dropout=0.3):
#         super().__init__()
#         self.num_experts = num_experts

#         self.encoder = nn.Sequential(
#             nn.Linear(65536, 2048),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(2048, 1024),
#             nn.ReLU(),
#             nn.Dropout(dropout)
#         )

#         self.experts = nn.ModuleList([
#             self._build_expert(1024, hidden_dims, dropout)
#             for _ in range(num_experts)
#         ])

#         self.gate = nn.Sequential(
#             nn.Linear(1024, 256),
#             nn.ReLU(),
#             nn.Linear(256, num_experts),
#             nn.Softmax(dim=-1)
#         )

#     def _build_expert(self, input_dim, hidden_dims, dropout):
#         layers = []
#         prev_dim = input_dim
#         for hidden_dim in hidden_dims:
#             layers.extend([nn.Linear(prev_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
#             prev_dim = hidden_dim
#         layers.append(nn.Linear(prev_dim, 1))
#         return nn.Sequential(*layers)

#     def forward(self, x):
#         feat = self.encoder(x)
#         exp_preds = torch.stack([e(feat).squeeze(-1) for e in self.experts], dim=-1)
#         weights = self.gate(feat)
#         return (exp_preds * weights).sum(dim=-1)

# class HierarchicalFusion(nn.Module):
#     def __init__(self, num_layers=4, nhead=4, dropout=0.2):
#         super().__init__()
#         self.mod_emb = nn.Embedding(8, 128)

#         encoder_layer = nn.TransformerEncoderLayer(
#             d_model=256,
#             nhead=nhead,
#             dim_feedforward=1024,
#             dropout=dropout,
#             batch_first=True
#         )
#         self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

#         self.head = nn.Sequential(
#             nn.LayerNorm(256),
#             nn.Linear(256, 256),
#             nn.GELU(),
#             nn.Dropout(dropout),
#             nn.Linear(256, 128),
#             nn.GELU(),
#             nn.Linear(128, 1)
#         )

#     def forward(self, x):
#         B = x.shape[0]
#         mod_tokens = x.reshape(B, 8, 64, 128)
#         mod_mean = mod_tokens.mean(dim=2)
#         mod_max = mod_tokens.max(dim=2)[0]
#         mod_embeds = torch.cat([mod_mean, mod_max], dim=-1)

#         mod_ids = torch.arange(8, device=x.device)
#         type_emb = self.mod_emb(mod_ids).unsqueeze(0).expand(B, -1, -1).repeat(1, 1, 2)
#         mod_embeds = mod_embeds + type_emb

#         fused = self.transformer(mod_embeds)
#         return self.head(fused.mean(dim=1)).squeeze(-1)

# # =====================================================
# # EVALUATION FUNCTIONS
# # =====================================================

# def evaluate_predictions(y_true, y_pred, target_range):
#     """Compute all metrics for a set of predictions"""
#     mae = np.abs(y_pred - y_true).mean()
#     rmse = np.sqrt(((y_pred - y_true)**2).mean())
#     r2 = r2_score(y_true, y_pred)

#     pred_range = y_pred.max() - y_pred.min()
#     coverage = (pred_range / target_range) * 100
#     std_ratio = y_pred.std() / y_true.std()

#     return {
#         'mae': mae,
#         'rmse': rmse,
#         'r2': r2,
#         'coverage': coverage,
#         'std_ratio': std_ratio,
#         'pred_range': pred_range
#     }

# def compute_composite_score(metrics):
#     """Compute weighted composite score (lower is better for MAE/RMSE)"""
#     # Normalize metrics to [0, 1] where lower is better
#     mae_norm = metrics['mae'] / 20.0  # Assume max MAE ~ 20m
#     coverage_norm = 1.0 - (metrics['coverage'] / 100.0)  # Invert coverage
#     r2_norm = 1.0 - max(0, metrics['r2'])  # Invert R2

#     score = (COMPOSITE_WEIGHTS['mae'] * mae_norm +
#              COMPOSITE_WEIGHTS['coverage'] * coverage_norm +
#              COMPOSITE_WEIGHTS['r2'] * r2_norm)

#     return score

# def select_best_config(results, metric):
#     """Select best hyperparameter configuration"""
#     if metric == "mae":
#         return min(results, key=lambda x: x['metrics']['mae'])
#     elif metric == "rmse":
#         return min(results, key=lambda x: x['metrics']['rmse'])
#     elif metric == "coverage":
#         return max(results, key=lambda x: x['metrics']['coverage'])
#     elif metric == "composite":
#         return min(results, key=lambda x: compute_composite_score(x['metrics']))
#     else:
#         raise ValueError(f"Unknown metric: {metric}")

# # =====================================================
# # TRAINING FUNCTIONS
# # =====================================================

# def train_pytorch_model(model, train_loader, val_loader, lr=1e-3, epochs=100, patience=25):
#     """Train a PyTorch model with early stopping"""
#     model.to(DEVICE)
#     optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
#     criterion = nn.L1Loss()

#     best_val_mae = float('inf')
#     patience_counter = 0

#     for epoch in range(epochs):
#         # Train
#         model.train()
#         for X_batch, y_batch in train_loader:
#             X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
#             optimizer.zero_grad()
#             preds = model(X_batch)
#             loss = criterion(preds, y_batch)
#             loss.backward()
#             torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#             optimizer.step()

#         # Validate
#         model.eval()
#         val_preds = []
#         with torch.no_grad():
#             for X_batch, y_batch in val_loader:
#                 X_batch = X_batch.to(DEVICE)
#                 preds = model(X_batch)
#                 val_preds.append(preds.cpu())

#         val_preds = torch.cat(val_preds).numpy()
#         val_mae = np.abs(val_preds - y_val).mean()

#         if val_mae < best_val_mae:
#             best_val_mae = val_mae
#             patience_counter = 0
#             best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
#         else:
#             patience_counter += 1
#             if patience_counter >= patience:
#                 break

#     # Restore best model
#     model.load_state_dict(best_model_state)
#     return model

# # =====================================================
# # HYPERPARAMETER TUNING - SKLEARN MODELS
# # =====================================================

# print("\n" + "="*70)
# print("PHASE 1: HYPERPARAMETER TUNING (Validation Set)")
# print("="*70)

# tuning_results = {}

# # 1. RF Raw Baseline
# print("\n[1/8] Tuning RF_Raw_Baseline...")
# param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["RF_Raw_Baseline"])
# results = []
# for i, params in enumerate(param_grid):
#     if (i+1) % 10 == 0:
#         print(f"  Config {i+1}/{len(param_grid)}")

#     rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=-1)
#     rf.fit(X_train, y_train)
#     pred_val = rf.predict(X_val)
#     metrics = evaluate_predictions(y_val, pred_val, target_range_val)
#     results.append({'params': params, 'metrics': metrics})

# best_config = select_best_config(results, TUNING_METRIC)
# tuning_results["RF_Raw_Baseline"] = best_config
# print(f"  ✓ Best: {best_config['params']}")
# print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# # 2. Ridge
# print("\n[2/8] Tuning Ridge...")
# param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["Ridge"])
# results = []
# for params in param_grid:
#     ridge = Ridge(**params)
#     ridge.fit(X_train, y_train)
#     pred_val = ridge.predict(X_val)
#     metrics = evaluate_predictions(y_val, pred_val, target_range_val)
#     results.append({'params': params, 'metrics': metrics})

# best_config = select_best_config(results, TUNING_METRIC)
# tuning_results["Ridge"] = best_config
# print(f"  ✓ Best: {best_config['params']}")
# print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# # 3. Lasso
# print("\n[3/8] Tuning Lasso...")
# param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["Lasso"])
# results = []
# for params in param_grid:
#     lasso = Lasso(**params, max_iter=5000)
#     lasso.fit(X_train, y_train)
#     pred_val = lasso.predict(X_val)
#     metrics = evaluate_predictions(y_val, pred_val, target_range_val)
#     results.append({'params': params, 'metrics': metrics})

# best_config = select_best_config(results, TUNING_METRIC)
# tuning_results["Lasso"] = best_config
# print(f"  ✓ Best: {best_config['params']}")
# print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# # 4. ElasticNet
# print("\n[4/8] Tuning ElasticNet...")
# param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["ElasticNet"])
# results = []
# for params in param_grid:
#     enet = ElasticNet(**params, max_iter=5000)
#     enet.fit(X_train, y_train)
#     pred_val = enet.predict(X_val)
#     metrics = evaluate_predictions(y_val, pred_val, target_range_val)
#     results.append({'params': params, 'metrics': metrics})

# best_config = select_best_config(results, TUNING_METRIC)
# tuning_results["ElasticNet"] = best_config
# print(f"  ✓ Best: {best_config['params']}")
# print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# # 5. RF Embeddings
# print("\n[5/8] Tuning RF_Embeddings...")
# param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["RF_Embeddings"])
# results = []
# for i, params in enumerate(param_grid):
#     if (i+1) % 10 == 0:
#         print(f"  Config {i+1}/{len(param_grid)}")

#     rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=-1)
#     rf.fit(X_train, y_train)
#     pred_val = rf.predict(X_val)
#     metrics = evaluate_predictions(y_val, pred_val, target_range_val)
#     results.append({'params': params, 'metrics': metrics})

# best_config = select_best_config(results, TUNING_METRIC)
# tuning_results["RF_Embeddings"] = best_config
# print(f"  ✓ Best: {best_config['params']}")
# print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# # =====================================================
# # HYPERPARAMETER TUNING - PYTORCH MODELS
# # =====================================================

# train_loader = DataLoader(SimpleDataset(X_train, y_train), batch_size=32, shuffle=True, num_workers=0)
# val_loader = DataLoader(SimpleDataset(X_val, y_val), batch_size=32, shuffle=False, num_workers=0)

# # 6. SimpleMLP
# print("\n[6/8] Tuning SimpleMLP...")
# param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["SimpleMLP"])
# results = []
# for i, params in enumerate(param_grid):
#     print(f"  Config {i+1}/{len(param_grid)}: hidden={params['hidden_dims']}, lr={params['lr']}, dropout={params['dropout']}")

#     model = SimpleMLP(hidden_dims=params['hidden_dims'], dropout=params['dropout'])
#     model = train_pytorch_model(model, train_loader, val_loader, lr=params['lr'], epochs=50, patience=15)

#     model.eval()
#     with torch.no_grad():
#         pred_val = model(torch.FloatTensor(X_val).to(DEVICE)).cpu().numpy()

#     metrics = evaluate_predictions(y_val, pred_val, target_range_val)
#     results.append({'params': params, 'metrics': metrics})

#     print(f"    Val MAE: {metrics['mae']:.2f}m, Coverage: {metrics['coverage']:.1f}%")

# best_config = select_best_config(results, TUNING_METRIC)
# tuning_results["SimpleMLP"] = best_config
# print(f"  ✓ Best: hidden={best_config['params']['hidden_dims']}, lr={best_config['params']['lr']}")
# print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# # 7. MoE
# print("\n[7/8] Tuning MoE...")
# param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["MoE"])
# results = []
# for i, params in enumerate(param_grid):
#     print(f"  Config {i+1}/{len(param_grid)}: experts={params['num_experts']}, lr={params['lr']}")

#     model = MixtureOfExperts(
#         num_experts=params['num_experts'],
#         hidden_dims=params['hidden_dims'],
#         dropout=params['dropout']
#     )
#     model = train_pytorch_model(model, train_loader, val_loader, lr=params['lr'], epochs=50, patience=15)

#     model.eval()
#     with torch.no_grad():
#         pred_val = model(torch.FloatTensor(X_val).to(DEVICE)).cpu().numpy()

#     metrics = evaluate_predictions(y_val, pred_val, target_range_val)
#     results.append({'params': params, 'metrics': metrics})

#     print(f"    Val MAE: {metrics['mae']:.2f}m, Coverage: {metrics['coverage']:.1f}%")

# best_config = select_best_config(results, TUNING_METRIC)
# tuning_results["MoE"] = best_config
# print(f"  ✓ Best: experts={best_config['params']['num_experts']}, lr={best_config['params']['lr']}")
# print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# # 8. Hierarchical
# print("\n[8/8] Tuning Hierarchical...")
# param_grid = ParameterGrid(HYPERPARAMETER_GRIDS["Hierarchical"])
# results = []
# for i, params in enumerate(param_grid):
#     print(f"  Config {i+1}/{len(param_grid)}: layers={params['num_layers']}, nhead={params['nhead']}, lr={params['lr']}")

#     model = HierarchicalFusion(
#         num_layers=params['num_layers'],
#         nhead=params['nhead'],
#         dropout=params['dropout']
#     )
#     model = train_pytorch_model(model, train_loader, val_loader, lr=params['lr'], epochs=50, patience=15)

#     model.eval()
#     with torch.no_grad():
#         pred_val = model(torch.FloatTensor(X_val).to(DEVICE)).cpu().numpy()

#     metrics = evaluate_predictions(y_val, pred_val, target_range_val)
#     results.append({'params': params, 'metrics': metrics})

#     print(f"    Val MAE: {metrics['mae']:.2f}m, Coverage: {metrics['coverage']:.1f}%")

# best_config = select_best_config(results, TUNING_METRIC)
# tuning_results["Hierarchical"] = best_config
# print(f"  ✓ Best: layers={best_config['params']['num_layers']}, nhead={best_config['params']['nhead']}")
# print(f"    Val MAE: {best_config['metrics']['mae']:.2f}m, Coverage: {best_config['metrics']['coverage']:.1f}%")

# # =====================================================
# # SAVE TUNING RESULTS
# # =====================================================

# print("\n" + "="*70)
# print("TUNING COMPLETE - SAVING RESULTS")
# print("="*70)

# tuning_summary = []
# for model_name, config in tuning_results.items():
#     tuning_summary.append({
#         'Model': model_name,
#         'Best_Params': str(config['params']),
#         'Val_MAE': config['metrics']['mae'],
#         'Val_Coverage': config['metrics']['coverage'],
#         'Val_RMSE': config['metrics']['rmse'],
#         'Val_R2': config['metrics']['r2']
#     })

# df_tuning = pd.DataFrame(tuning_summary)
# tuning_path = os.path.join(OUTPUT_DIR, "Stage2_Bakeoff_Tuning_Results.csv")
# df_tuning.to_csv(tuning_path, index=False)
# print(f"\n✓ Saved tuning results: {tuning_path}")

# print("\n" + "="*70)
# print("BEST HYPERPARAMETERS SELECTED")
# print("="*70)
# print(df_tuning.to_string(index=False))

# # Save full tuning results as pickle
# tuning_pickle_path = os.path.join(OUTPUT_DIR, "Stage2_Bakeoff_Tuning_Full.pkl")
# with open(tuning_pickle_path, 'wb') as f:
#     pickle.dump(tuning_results, f)
# print(f"\n✓ Saved full tuning data: {tuning_pickle_path}")

# print("\n" + "="*70)
# print("PHASE 1 COMPLETE - Ready for Phase 2 (Final Training)")
# print("="*70)
# print("\nNext: Run Phase 2 script to train models with best hyperparameters on test set")

Cell by cell break down

In [ ]:
# =====================================================
# CELL 1: SETUP (Run this first)
# =====================================================

import os, sys, time, pickle, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("="*70)
print("STAGE 2 BAKE-OFF - SETUP")
print("="*70)

# Paths
DATA_DIR = "/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/Embedding_Checkpoints_Stage1"
OUTPUT_DIR = "/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/Stage2_Bakeoff_Results"
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "bakeoff_progress.pkl")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load data
print("\nLoading data...")
dataset_train = torch.load(f"{DATA_DIR}/dataset_train.pt", weights_only=False)
dataset_test = torch.load(f"{DATA_DIR}/dataset_test.pt", weights_only=False)
dataset_val = torch.load(f"{DATA_DIR}/dataset_val.pt", weights_only=False)

def extract_data(dataset):
    X, y = [], []
    for s in dataset:
        X.append(s['embeddings'].flatten().numpy())
        y.append(s['target'].item())
    return np.array(X), np.array(y)

X_train, y_train = extract_data(dataset_train)
X_test, y_test = extract_data(dataset_test)
X_val, y_val = extract_data(dataset_val)

print(f"✓ Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"✓ Device: {DEVICE}")

target_range_val = y_val.max() - y_val.min()
target_range_test = y_test.max() - y_test.min()

# Evaluation function
def evaluate_predictions(y_true, y_pred, target_range):
    mae = np.abs(y_pred - y_true).mean()
    rmse = np.sqrt(((y_pred - y_true)**2).mean())
    r2 = r2_score(y_true, y_pred)
    pred_range = y_pred.max() - y_pred.min()
    coverage = (pred_range / target_range) * 100
    std_ratio = y_pred.std() / y_true.std()
    return {
        'mae': mae, 'rmse': rmse, 'r2': r2,
        'coverage': coverage, 'std_ratio': std_ratio, 'pred_range': pred_range
    }

# Load or initialize progress
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, 'rb') as f:
        tuning_results = pickle.load(f)
    print(f"\n✓ Loaded progress: {len(tuning_results)} models completed")
else:
    tuning_results = {}
    print("\n✓ Starting fresh")

def save_progress():
    with open(PROGRESS_FILE, 'wb') as f:
        pickle.dump(tuning_results, f)
    print(f"  💾 Progress saved to {PROGRESS_FILE}")

print("\n✅ SETUP COMPLETE - Ready to tune models")
print("="*70)

STAGE 2 BAKE-OFF - SETUP

Loading data...
✓ Train: (679, 65536), Val: (86, 65536), Test: (152, 65536)
✓ Device: cpu

✓ Starting fresh

✅ SETUP COMPLETE - Ready to tune models


In [ ]:
# =====================================================
# CELL 2: RF Raw Baseline (15-20 min)
# =====================================================

if "RF_Raw_Baseline" in tuning_results:
    print("⏭️  RF_Raw_Baseline already tuned - SKIPPING")
else:
    print("\n[1/8] Tuning RF_Raw_Baseline...")
    print("Expected time: 15-20 minutes")

    # REDUCED grid for speed (was 144, now 36)
    param_grid = ParameterGrid({
        'n_estimators': [100, 200],
        'max_depth': [10, 15],  # Reduced from [5,10,15,20]
        'min_samples_split': [2, 5],  # Reduced from [2,5,10]
        'min_samples_leaf': [1, 2]  # Reduced from [1,2,4]
    })

    results = []
    start_time = time.time()

    for i, params in enumerate(param_grid):
        if (i+1) % 5 == 0:
            elapsed = time.time() - start_time
            est_total = elapsed / (i+1) * len(param_grid)
            remaining = est_total - elapsed
            print(f"  Config {i+1}/{len(param_grid)} - Est. {remaining/60:.1f} min remaining")

        rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=-1)
        rf.fit(X_train, y_train)
        pred_val = rf.predict(X_val)
        metrics = evaluate_predictions(y_val, pred_val, target_range_val)
        results.append({'params': params, 'metrics': metrics})

    # Select best by coverage
    best = max(results, key=lambda x: x['metrics']['coverage'])
    tuning_results["RF_Raw_Baseline"] = best

    print(f"  ✓ DONE in {(time.time()-start_time)/60:.1f} min")
    print(f"  Best: {best['params']}")
    print(f"  Val Coverage: {best['metrics']['coverage']:.1f}%, MAE: {best['metrics']['mae']:.2f}m")

    save_progress()



[1/8] Tuning RF_Raw_Baseline...
Expected time: 15-20 minutes


In [ ]:
# =====================================================
# CELL 3: Ridge (1-2 min)
# =====================================================

if "Ridge" in tuning_results:
    print("⏭️  Ridge already tuned - SKIPPING")
else:
    print("\n[2/8] Tuning Ridge...")
    print("Expected time: 1-2 minutes")

    param_grid = ParameterGrid({'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]})
    results = []
    start_time = time.time()

    for params in param_grid:
        ridge = Ridge(**params)
        ridge.fit(X_train, y_train)
        pred_val = ridge.predict(X_val)
        metrics = evaluate_predictions(y_val, pred_val, target_range_val)
        results.append({'params': params, 'metrics': metrics})

    best = max(results, key=lambda x: x['metrics']['coverage'])
    tuning_results["Ridge"] = best

    print(f"  ✓ DONE in {(time.time()-start_time):.1f} sec")
    print(f"  Best: {best['params']}")
    print(f"  Val Coverage: {best['metrics']['coverage']:.1f}%, MAE: {best['metrics']['mae']:.2f}m")

    save_progress()

In [ ]:
# =====================================================
# CELL 4: Lasso (2-3 min)
# =====================================================

if "Lasso" in tuning_results:
    print("⏭️  Lasso already tuned - SKIPPING")
else:
    print("\n[3/8] Tuning Lasso...")
    print("Expected time: 2-3 minutes")

    param_grid = ParameterGrid({'alpha': [0.001, 0.01, 0.1, 1.0, 10.0]})
    results = []
    start_time = time.time()

    for params in param_grid:
        lasso = Lasso(**params, max_iter=5000)
        lasso.fit(X_train, y_train)
        pred_val = lasso.predict(X_val)
        metrics = evaluate_predictions(y_val, pred_val, target_range_val)
        results.append({'params': params, 'metrics': metrics})

    best = max(results, key=lambda x: x['metrics']['coverage'])
    tuning_results["Lasso"] = best

    print(f"  ✓ DONE in {(time.time()-start_time):.1f} sec")
    print(f"  Best: {best['params']}")
    print(f"  Val Coverage: {best['metrics']['coverage']:.1f}%, MAE: {best['metrics']['mae']:.2f}m")

    save_progress()

In [ ]:
# =====================================================
# CELL 5: ElasticNet (3-5 min)
# =====================================================

if "ElasticNet" in tuning_results:
    print("⏭️  ElasticNet already tuned - SKIPPING")
else:
    print("\n[4/8] Tuning ElasticNet...")
    print("Expected time: 3-5 minutes")

    param_grid = ParameterGrid({
        'alpha': [0.1, 1.0, 10.0],
        'l1_ratio': [0.1, 0.5, 0.9]  # Reduced from [0.1,0.3,0.5,0.7,0.9]
    })
    results = []
    start_time = time.time()

    for params in param_grid:
        enet = ElasticNet(**params, max_iter=5000)
        enet.fit(X_train, y_train)
        pred_val = enet.predict(X_val)
        metrics = evaluate_predictions(y_val, pred_val, target_range_val)
        results.append({'params': params, 'metrics': metrics})

    best = max(results, key=lambda x: x['metrics']['coverage'])
    tuning_results["ElasticNet"] = best

    print(f"  ✓ DONE in {(time.time()-start_time):.1f} sec")
    print(f"  Best: {best['params']}")
    print(f"  Val Coverage: {best['metrics']['coverage']:.1f}%, MAE: {best['metrics']['mae']:.2f}m")

    save_progress()

In [ ]:
# =====================================================
# CELL 6: RF Embeddings (15-20 min)
# =====================================================

if "RF_Embeddings" in tuning_results:
    print("⏭️  RF_Embeddings already tuned - SKIPPING")
else:
    print("\n[5/8] Tuning RF_Embeddings...")
    print("Expected time: 15-20 minutes")

    param_grid = ParameterGrid({
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, 15],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    })
    results = []
    start_time = time.time()

    for i, params in enumerate(param_grid):
        if (i+1) % 5 == 0:
            elapsed = time.time() - start_time
            est_total = elapsed / (i+1) * len(param_grid)
            remaining = est_total - elapsed
            print(f"  Config {i+1}/{len(param_grid)} - Est. {remaining/60:.1f} min remaining")

        rf = RandomForestRegressor(**params, random_state=SEED, n_jobs=-1)
        rf.fit(X_train, y_train)
        pred_val = rf.predict(X_val)
        metrics = evaluate_predictions(y_val, pred_val, target_range_val)
        results.append({'params': params, 'metrics': metrics})

    best = max(results, key=lambda x: x['metrics']['coverage'])
    tuning_results["RF_Embeddings"] = best

    print(f"  ✓ DONE in {(time.time()-start_time)/60:.1f} min")
    print(f"  Best: {best['params']}")
    print(f"  Val Coverage: {best['metrics']['coverage']:.1f}%, MAE: {best['metrics']['mae']:.2f}m")

    save_progress()

In [ ]:
# =====================================================
# CELL 7: SimpleMLP (20-30 min)
# =====================================================

if "SimpleMLP" in tuning_results:
    print("⏭️  SimpleMLP already tuned - SKIPPING")
else:
    print("\n[6/8] Tuning SimpleMLP...")
    print("Expected time: 20-30 minutes")

    class SimpleDataset(Dataset):
        def __init__(self, X, y):
            self.X = torch.FloatTensor(X)
            self.y = torch.FloatTensor(y)
        def __len__(self): return len(self.X)
        def __getitem__(self, idx): return self.X[idx], self.y[idx]

    class SimpleMLP(nn.Module):
        def __init__(self, hidden_dims, dropout):
            super().__init__()
            layers = []
            prev_dim = 65536
            for h in hidden_dims:
                layers.extend([nn.Linear(prev_dim, h), nn.ReLU(), nn.Dropout(dropout)])
                prev_dim = h
            layers.append(nn.Linear(prev_dim, 1))
            self.net = nn.Sequential(*layers)
        def forward(self, x): return self.net(x).squeeze(-1)

    def train_model(model, train_loader, val_loader, lr, epochs=30, patience=10):
        model.to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.L1Loss()
        best_val_mae = float('inf')
        patience_counter = 0

        for epoch in range(epochs):
            model.train()
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(X_batch), y_batch)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            model.eval()
            val_preds = []
            with torch.no_grad():
                for X_batch, _ in val_loader:
                    val_preds.append(model(X_batch.to(DEVICE)).cpu())
            val_preds = torch.cat(val_preds).numpy()
            val_mae = np.abs(val_preds - y_val).mean()

            if val_mae < best_val_mae:
                best_val_mae = val_mae
                patience_counter = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break

        model.load_state_dict(best_state)
        return model

    train_loader = DataLoader(SimpleDataset(X_train, y_train), batch_size=32, shuffle=True, num_workers=0)
    val_loader = DataLoader(SimpleDataset(X_val, y_val), batch_size=32, shuffle=False, num_workers=0)

    # REDUCED grid (was 27, now 6)
    param_grid = ParameterGrid({
        'hidden_dims': [[2048, 512, 128], [2048, 1024, 512, 128]],
        'dropout': [0.3],  # Reduced from [0.2,0.3,0.4]
        'lr': [5e-4, 1e-3, 2e-3]
    })
    results = []
    start_time = time.time()

    for i, params in enumerate(param_grid):
        config_start = time.time()
        print(f"  Config {i+1}/{len(param_grid)}: {params}")

        model = SimpleMLP(params['hidden_dims'], params['dropout'])
        model = train_model(model, train_loader, val_loader, params['lr'])

        model.eval()
        with torch.no_grad():
            pred_val = model(torch.FloatTensor(X_val).to(DEVICE)).cpu().numpy()

        metrics = evaluate_predictions(y_val, pred_val, target_range_val)
        results.append({'params': params, 'metrics': metrics})

        config_time = (time.time() - config_start) / 60
        remaining = config_time * (len(param_grid) - i - 1)
        print(f"    Val Coverage: {metrics['coverage']:.1f}%, MAE: {metrics['mae']:.2f}m ({config_time:.1f} min, {remaining:.1f} min left)")

    best = max(results, key=lambda x: x['metrics']['coverage'])
    tuning_results["SimpleMLP"] = best

    print(f"  ✓ DONE in {(time.time()-start_time)/60:.1f} min")
    print(f"  Best: {best['params']}")
    print(f"  Val Coverage: {best['metrics']['coverage']:.1f}%, MAE: {best['metrics']['mae']:.2f}m")

    save_progress()


In [17]:
# =====================================================
# CELL 8: MoE (30-45 min)
# =====================================================

if "MoE" in tuning_results:
    print("⏭️  MoE already tuned - SKIPPING")
else:
    print("\n[7/8] Tuning MoE...")
    print("Expected time: 30-45 minutes")
    print("⚠️  This is the slowest model - be patient!")

    class MixtureOfExperts(nn.Module):
        def __init__(self, num_experts, hidden_dims, dropout):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Linear(65536, 2048), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(2048, 1024), nn.ReLU(), nn.Dropout(dropout))
            self.experts = nn.ModuleList([
                nn.Sequential(
                    nn.Linear(1024, hidden_dims[0]), nn.ReLU(), nn.Dropout(dropout),
                    nn.Linear(hidden_dims[0], hidden_dims[1]), nn.ReLU(),
                    nn.Linear(hidden_dims[1], 1))
                for _ in range(num_experts)])
            self.gate = nn.Sequential(
                nn.Linear(1024, 256), nn.ReLU(),
                nn.Linear(256, num_experts), nn.Softmax(dim=-1))
        def forward(self, x):
            feat = self.encoder(x)
            exp_preds = torch.stack([e(feat).squeeze(-1) for e in self.experts], dim=-1)
            return (exp_preds * self.gate(feat)).sum(dim=-1)

    # REDUCED grid (was 24, now 4)
    param_grid = ParameterGrid({
        'num_experts': [3, 5],  # Reduced from [3,5,7]
        'hidden_dims': [[512, 256]],  # Reduced from [[1024,512,256], [2048,512,256]]
        'dropout': [0.3],  # Reduced from [0.2,0.3]
        'lr': [5e-4, 1e-3]
    })
    results = []
    start_time = time.time()

    for i, params in enumerate(param_grid):
        config_start = time.time()
        print(f"  Config {i+1}/{len(param_grid)}: {params}")

        model = MixtureOfExperts(params['num_experts'], params['hidden_dims'], params['dropout'])
        model = train_model(model, train_loader, val_loader, params['lr'])

        model.eval()
        with torch.no_grad():
            pred_val = model(torch.FloatTensor(X_val).to(DEVICE)).cpu().numpy()

        metrics = evaluate_predictions(y_val, pred_val, target_range_val)
        results.append({'params': params, 'metrics': metrics})

        config_time = (time.time() - config_start) / 60
        remaining = config_time * (len(param_grid) - i - 1)
        print(f"    Val Coverage: {metrics['coverage']:.1f}%, MAE: {metrics['mae']:.2f}m ({config_time:.1f} min, {remaining:.1f} min left)")

    best = max(results, key=lambda x: x['metrics']['coverage'])
    tuning_results["MoE"] = best

    print(f"  ✓ DONE in {(time.time()-start_time)/60:.1f} min")
    print(f"  Best: {best['params']}")
    print(f"  Val Coverage: {best['metrics']['coverage']:.1f}%, MAE: {best['metrics']['mae']:.2f}m")

    save_progress()


[7/8] Tuning MoE...
Expected time: 30-45 minutes
⚠️  This is the slowest model - be patient!
  Config 1/4: {'dropout': 0.3, 'hidden_dims': [512, 256], 'lr': 0.0005, 'num_experts': 3}
    Val Coverage: 47.1%, MAE: 5.94m (31.2 min, 93.6 min left)
  Config 2/4: {'dropout': 0.3, 'hidden_dims': [512, 256], 'lr': 0.0005, 'num_experts': 5}


KeyboardInterrupt: 

In [13]:
# =====================================================
# CELL 9: Hierarchical (25-40 min)
# =====================================================

if "Hierarchical" in tuning_results:
    print("⏭️  Hierarchical already tuned - SKIPPING")
else:
    print("\n[8/8] Tuning Hierarchical...")
    print("Expected time: 25-40 minutes")

    class HierarchicalFusion(nn.Module):
        def __init__(self, num_layers, nhead, dropout):
            super().__init__()
            self.mod_emb = nn.Embedding(8, 128)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=256, nhead=nhead, dim_feedforward=1024,
                dropout=dropout, batch_first=True)
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.head = nn.Sequential(
                nn.LayerNorm(256), nn.Linear(256, 256), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(256, 128), nn.GELU(), nn.Linear(128, 1))
        def forward(self, x):
            B = x.shape[0]
            mod_tokens = x.reshape(B, 8, 64, 128)
            mod_embeds = torch.cat([mod_tokens.mean(dim=2), mod_tokens.max(dim=2)[0]], dim=-1)
            mod_ids = torch.arange(8, device=x.device)
            type_emb = self.mod_emb(mod_ids).unsqueeze(0).expand(B, -1, -1).repeat(1, 1, 2)
            mod_embeds = mod_embeds + type_emb
            fused = self.transformer(mod_embeds)
            return self.head(fused.mean(dim=1)).squeeze(-1)

    # REDUCED grid (was 24, now 4)
    param_grid = ParameterGrid({
        'num_layers': [2, 4],  # Reduced from [2,4,6]
        'nhead': [4],  # Reduced from [4,8]
        'dropout': [0.2],  # Reduced from [0.1,0.2,0.3]
        'lr': [5e-4, 1e-3]
    })
    results = []
    start_time = time.time()

    for i, params in enumerate(param_grid):
        config_start = time.time()
        print(f"  Config {i+1}/{len(param_grid)}: {params}")

        model = HierarchicalFusion(params['num_layers'], params['nhead'], params['dropout'])
        model = train_model(model, train_loader, val_loader, params['lr'])

        model.eval()
        with torch.no_grad():
            pred_val = model(torch.FloatTensor(X_val).to(DEVICE)).cpu().numpy()

        metrics = evaluate_predictions(y_val, pred_val, target_range_val)
        results.append({'params': params, 'metrics': metrics})

        config_time = (time.time() - config_start) / 60
        remaining = config_time * (len(param_grid) - i - 1)
        print(f"    Val Coverage: {metrics['coverage']:.1f}%, MAE: {metrics['mae']:.2f}m ({config_time:.1f} min, {remaining:.1f} min left)")

    best = max(results, key=lambda x: x['metrics']['coverage'])
    tuning_results["Hierarchical"] = best

    print(f"  ✓ DONE in {(time.time()-start_time)/60:.1f} min")
    print(f"  Best: {best['params']}")
    print(f"  Val Coverage: {best['metrics']['coverage']:.1f}%, MAE: {best['metrics']['mae']:.2f}m")

    save_progress()



[8/8] Tuning Hierarchical...
Expected time: 25-40 minutes
  Config 1/4: {'dropout': 0.2, 'lr': 0.0005, 'nhead': 4, 'num_layers': 2}
    Val Coverage: 30.7%, MAE: 6.14m (0.5 min, 1.6 min left)
  Config 2/4: {'dropout': 0.2, 'lr': 0.0005, 'nhead': 4, 'num_layers': 4}
    Val Coverage: 34.5%, MAE: 5.78m (1.4 min, 2.7 min left)
  Config 3/4: {'dropout': 0.2, 'lr': 0.001, 'nhead': 4, 'num_layers': 2}
    Val Coverage: 29.6%, MAE: 6.58m (1.2 min, 1.2 min left)
  Config 4/4: {'dropout': 0.2, 'lr': 0.001, 'nhead': 4, 'num_layers': 4}
    Val Coverage: 0.0%, MAE: 6.52m (0.9 min, 0.0 min left)
  ✓ DONE in 4.0 min
  Best: {'dropout': 0.2, 'lr': 0.0005, 'nhead': 4, 'num_layers': 4}
  Val Coverage: 34.5%, MAE: 5.78m
  💾 Progress saved to /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/Stage2_Bakeoff_Results/bakeoff_progress.pkl


In [16]:
# =====================================================
# CELL 10: SUMMARY
# =====================================================

print("\n" + "="*70)
print("TUNING COMPLETE!")
print("="*70)

summary = []
for model_name, config in tuning_results.items():
    summary.append({
        'Model': model_name,
        'Best_Params': str(config['params']),
        'Val_MAE': config['metrics']['mae'],
        'Val_Coverage': config['metrics']['coverage'],
        'Val_RMSE': config['metrics']['rmse'],
        'Val_R2': config['metrics']['r2']
    })

df_summary = pd.DataFrame(summary).sort_values('Val_Coverage', ascending=False)
print("\n" + df_summary.to_string(index=False))

# Save final results
csv_path = os.path.join(OUTPUT_DIR, "Stage2_Bakeoff_Tuning_Results.csv")
df_summary.to_csv(csv_path, index=False)
print(f"\n✓ Saved: {csv_path}")

print("\n✅ ALL MODELS TUNED!")
print("="*70)


TUNING COMPLETE!

       Model                                                     Best_Params  Val_MAE  Val_Coverage  Val_RMSE    Val_R2
       Lasso                                                {'alpha': 0.001} 6.475510    128.342137  7.877838 -0.155955
       Ridge                                                 {'alpha': 0.01} 5.533962    102.676610  6.683593  0.167955
  ElasticNet                                 {'alpha': 0.1, 'l1_ratio': 0.1} 5.566704     71.817592  6.649166  0.176504
   SimpleMLP {'dropout': 0.3, 'hidden_dims': [2048, 512, 128], 'lr': 0.0005} 6.071232     50.662556  7.795751 -0.131991
Hierarchical     {'dropout': 0.2, 'lr': 0.0005, 'nhead': 4, 'num_layers': 4} 5.778458     34.520490  7.545923 -0.060600

✓ Saved: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/Stage2_Bakeoff_Results/Stage2_Bakeoff_Tuning_Results.csv

✅ ALL MODELS TUNED!


run off the progress pickle

In [15]:
# =====================================================
# CHECK BAKE-OFF PROGRESS
# See which models completed and resume from there
# =====================================================

import pickle
import os

OUTPUT_DIR = "/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/Stage2_Bakeoff_Results"
PROGRESS_FILE = os.path.join(OUTPUT_DIR, "bakeoff_progress.pkl")

print("="*70)
print("BAKE-OFF PROGRESS CHECK")
print("="*70)

if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, 'rb') as f:
        tuning_results = pickle.load(f)

    print(f"\n✓ Found progress file: {PROGRESS_FILE}")
    print(f"✓ Models completed: {len(tuning_results)}/8\n")

    print("COMPLETED MODELS:")
    print("-" * 70)

    for i, (model_name, config) in enumerate(tuning_results.items(), 1):
        metrics = config['metrics']
        print(f"\n{i}. {model_name}")
        print(f"   Coverage: {metrics['coverage']:.1f}%")
        print(f"   MAE: {metrics['mae']:.2f}m")
        print(f"   RMSE: {metrics['rmse']:.2f}m")
        print(f"   R²: {metrics['r2']:.3f}")
        print(f"   Best params: {config['params']}")

    print("\n" + "="*70)
    print("REMAINING MODELS:")
    print("="*70)

    all_models = [
        # "RF_Raw_Baseline",
        "Ridge",
        "Lasso",
        "ElasticNet",
        # "RF_Embeddings",
        "SimpleMLP",
        "MoE",
        "Hierarchical"
    ]

    remaining = [m for m in all_models if m not in tuning_results]

    if remaining:
        print(f"\n{len(remaining)} models still need tuning:")
        for i, model_name in enumerate(remaining, 1):
            print(f"  {i}. {model_name}")

        print("\n" + "="*70)
        print("TO RESUME:")
        print("="*70)
        print("\nJust re-run the bake-off script!")
        print("It will automatically skip completed models and continue.")
        print("\nOR use the ULTRA_FAST version for faster completion.")
    else:
        print("\n✅ ALL MODELS COMPLETE!")
        print("\nYou can now run the summary/visualization script.")

else:
    print("\n✗ No progress file found")
    print(f"   Expected location: {PROGRESS_FILE}")
    print("\nEither:")
    print("  1. Bake-off hasn't started yet")
    print("  2. Progress file is in a different location")

print("="*70)

BAKE-OFF PROGRESS CHECK

✓ Found progress file: /content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/Stage2_Bakeoff_Results/bakeoff_progress.pkl
✓ Models completed: 5/8

COMPLETED MODELS:
----------------------------------------------------------------------

1. Ridge
   Coverage: 102.7%
   MAE: 5.53m
   RMSE: 6.68m
   R²: 0.168
   Best params: {'alpha': 0.01}

2. Lasso
   Coverage: 128.3%
   MAE: 6.48m
   RMSE: 7.88m
   R²: -0.156
   Best params: {'alpha': 0.001}

3. ElasticNet
   Coverage: 71.8%
   MAE: 5.57m
   RMSE: 6.65m
   R²: 0.177
   Best params: {'alpha': 0.1, 'l1_ratio': 0.1}

4. SimpleMLP
   Coverage: 50.7%
   MAE: 6.07m
   RMSE: 7.80m
   R²: -0.132
   Best params: {'dropout': 0.3, 'hidden_dims': [2048, 512, 128], 'lr': 0.0005}

5. Hierarchical
   Coverage: 34.5%
   MAE: 5.78m
   RMSE: 7.55m
   R²: -0.061
   Best params: {'dropout': 0.2, 'lr': 0.0005, 'nhead': 4, 'num_layers': 4}

REMAINING MODELS:

1 models still need tuning:
  1. MoE

TO RESUME:

Just re-run 